# SweetViz+ demo

SweetViz+ adds four things on top of SweetViz:

1. **Outlier detection**: IQR (one column at a time) and Isolation Forest (whole rows)
2. **A quality score** from 0 to 100, with a letter grade
3. **Plain-language cleaning recommendations**
4. **Auto-clean** of the safe issues, with a before/after score

SweetViz itself is used as it is and is not modified. Run the cells from top to bottom.

## 1. Setup

- **Jupyter on your own computer:** if you ran `pip install -e .` in the project folder, this cell just imports SweetViz+.
- **Google Colab:** this cell asks you to upload `sweetviz-plus.zip`, then installs it.

In [ ]:
import importlib
import subprocess
import sys
import zipfile

try:
    import sweetviz_plus as svp
except ImportError:
    try:
        from google.colab import files
    except ImportError:
        raise ImportError(
            "sweetviz_plus is not installed. In a terminal, go to the project folder and run: pip install -e ."
        )

    print("Choose sweetviz-plus.zip when the upload button appears.")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./sweetviz-plus"], check=True)
    importlib.invalidate_caches()
    import sweetviz_plus as svp

print("SweetViz+ version", svp.__version__)

## 2. Load the data

This uses the real Titanic dataset. If there is no internet connection, it falls back to the small sample file in the project.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

try:
    df = pd.read_csv(URL)
    print("Loaded the real Titanic dataset")
except Exception:
    df = pd.read_csv("sample_data/titanic_sample.csv")
    print("No internet, loaded the small sample file instead")

print("Shape:", df.shape)
df.head()

## 3. Analyze

One line runs the outlier detection, the quality score and the recommendations. `summary()` prints all of it.

In [ ]:
report = svp.analyze(df)
report.summary()

## 4. Look at the pieces

Everything in the summary is also available as data you can work with.

In [ ]:
report.outliers["iqr_results"].head(10)

In [ ]:
report.quality["breakdown"]

In [ ]:
for number, rec in enumerate(report.recommendations, start=1):
    print(f"{number}. {rec}")

## 5. Auto-clean

This only fixes the safe issues: it drops columns that are over 70% missing, removes exact duplicate rows, and caps IQR outliers in columns that are not highly skewed. Missing values and skew are left for you to decide.

`auto_clean()` returns a new DataFrame. Your original `df` is not changed.

In [ ]:
cleaned_df = report.auto_clean()

comparison = report.comparison
print(
    f"Quality score: {comparison['before_score']} ({comparison['before_grade']})"
    f"  ->  {comparison['after_score']} ({comparison['after_grade']})"
)
print()
for change in comparison["changes_made"]:
    print("-", change)
print()
print("Shape before:", df.shape, "  after:", cleaned_df.shape)

In [ ]:
cleaned_df.head()

## 6. The full SweetViz report

This is SweetViz's own report, shown inside the notebook. It can take a little while to build.

In [ ]:
report.show_notebook()

## 7. Try your own data

Replace the file name below with your own CSV and run the cell. In Colab, upload the file first using the folder icon in the left sidebar.

```python
my_df = pd.read_csv("your_file.csv")
my_report = svp.analyze(my_df)
my_report.summary()
```